In [13]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- Auto-detect device ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# --- Dummy Model ---
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = nn.Linear(16, 16)
        self.classifier = nn.Linear(16, 2)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids)
        return self.classifier(x)

    def forward_mixup(self, input_ids_x1, attention_mask_x1, input_ids_x2, attention_mask_x2, labels_x,
                      input_ids_u1, attention_mask_u1, input_ids_u2, attention_mask_u2, labels_u,
                      layer_index, mix_lambda, device):
        # Simple mixup in input space
        emb_x1 = self.bert(input_ids_x1)
        emb_x2 = self.bert(input_ids_x2)
        emb_u1 = self.bert(input_ids_u1)
        emb_u2 = self.bert(input_ids_u2)

        x_mix = mix_lambda * emb_x1 + (1 - mix_lambda) * emb_x2
        u_mix = mix_lambda * emb_u1 + (1 - mix_lambda) * emb_u2
        labels_mix = mix_lambda * labels_x + (1 - mix_lambda) * labels_u

        embeddings = torch.cat([x_mix, u_mix], dim=0)
        mixed_labels = torch.cat([labels_mix, labels_mix], dim=0)

        logits = self.classifier(embeddings)
        return logits, mixed_labels


# --- Dummy Dataset ---
class DummyDataset(Dataset):
    def __init__(self, size=8):
        self.size = size

    def __getitem__(self, idx):
        seq_len = 16
        return {
            'input_ids_1': torch.randn(seq_len),
            'input_ids_2': torch.randn(seq_len),
            'attention_mask_1': torch.ones(seq_len),
            'attention_mask_2': torch.ones(seq_len),
            'labels': torch.randint(0, 2, (1,)),         # scalar label
            'probability': torch.rand(1)                 # scalar prob
        }

    def __len__(self):
        return self.size

# --- Dummy SemiLoss ---
def dummy_semiloss(logits_x, targets_x, logits_u, targets_u, epoch_no, warmup_epochs):
    Lx = torch.mean((logits_x - targets_x) ** 2)
    Lu = torch.mean((logits_u - targets_u) ** 2)
    lambda_u = 1.0
    return Lx, Lu, lambda_u

# --- Collate Function (MPS-safe) ---
def collate_fn(batch):
    return {k: torch.stack([d[k] for d in batch]) for k in batch[0]}

# --- Train Function ---
def train(epoch_no, model1, model2, optimizer, semiloss, labelled_loader, unlabelled_loader, warmup_epochs, batch_size=64, temperature=0.5, alpha=0.5, num_class=2, device='cpu'):
    model1.train()
    model2.eval()

    unlabelled_train_iter = iter(unlabelled_loader)
    num_iter = (len(labelled_loader.dataset) // batch_size) + 1

    for batch_idx, batch in tqdm(enumerate(labelled_loader), desc="Training", total=num_iter):
        for k in batch:
            batch[k] = batch[k].to(device)

        input_ids_x1 = batch['input_ids_1']
        input_ids_x2 = batch['input_ids_2']
        attention_mask_x1 = batch['attention_mask_1']
        attention_mask_x2 = batch['attention_mask_2']
        labels_x = batch['labels'].squeeze(1).long()
        prob = batch['probability'].squeeze(1)

        try:
            batch_u = next(unlabelled_train_iter)
        except:
            unlabelled_train_iter = iter(unlabelled_loader)
            batch_u = next(unlabelled_train_iter)

        for k in batch_u:
            batch_u[k] = batch_u[k].to(device)

        input_ids_u1 = batch_u['input_ids_1']
        input_ids_u2 = batch_u['input_ids_2']
        attention_mask_u1 = batch_u['attention_mask_1']
        attention_mask_u2 = batch_u['attention_mask_2']

        batch_size = input_ids_x1.size(0)

        labels_x = torch.zeros(batch_size, 2, device=device).scatter_(1, labels_x.view(-1, 1), 1)
        prob = prob.view(-1, 1).float()

        with torch.no_grad():
            outputs_u11 = model1(input_ids_u1, attention_mask_u1)
            outputs_u12 = model1(input_ids_u2, attention_mask_u2)
            outputs_u21 = model2(input_ids_u1, attention_mask_u1)
            outputs_u22 = model2(input_ids_u2, attention_mask_u2)

            pu = (torch.softmax(outputs_u11, dim=1) + torch.softmax(outputs_u12, dim=1) +
                  torch.softmax(outputs_u21, dim=1) + torch.softmax(outputs_u22, dim=1)) / 4
            ptu = pu ** (1 / temperature)
            labels_u = ptu / ptu.sum(dim=1, keepdim=True)
            labels_u = labels_u.detach()

            outputs_x = model1(input_ids_x1, attention_mask_x1)
            outputs_x2 = model1(input_ids_x2, attention_mask_x2)

            px = (torch.softmax(outputs_x, dim=1) + torch.softmax(outputs_x2, dim=1)) / 2
            px = prob * labels_x + (1 - prob) * px
            ptx = px ** (1 / temperature)
            labels_x = ptx / ptx.sum(dim=1, keepdim=True)
            labels_x = labels_x.detach()

        l = np.random.beta(alpha, alpha)
        l = max(l, 1 - l)
        layer_index = np.random.choice([7, 9, 12]) - 1

        logits, mixed_labels = model1.forward_mixup(
            input_ids_x1, attention_mask_x1, input_ids_x2, attention_mask_x2, labels_x,
            input_ids_u1, attention_mask_u1, input_ids_u2, attention_mask_u2, labels_u,
            layer_index=layer_index, mix_lambda=l, device=device)

        logits_x = logits[:batch_size]
        targets_x = mixed_labels[:batch_size]
        logits_u = logits[batch_size:]
        targets_u = mixed_labels[batch_size:]

        Lx, Lu, lambda_u_val = semiloss(logits_x, targets_x, logits_u, targets_u, epoch_no, warmup_epochs)

        prior = torch.full((num_class,), 1 / num_class, device=device)
        pred_mean = torch.softmax(logits, dim=1).mean(0)
        penalty = torch.sum(prior * torch.log(prior / pred_mean))

        loss = Lx + lambda_u_val * Lu + penalty

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        grad_norms = [p.grad.norm().item() for p in model1.parameters() if p.grad is not None]
        near_zero_grad_count = sum(1 for norm in grad_norms if norm <= 1e-4)
        tqdm.write(f"Epoch {epoch_no}, Batch {batch_idx+1}/{num_iter}, Grad Norms near zero: {near_zero_grad_count}, "
                   f"Lx {Lx.item():.4f}, Lu {(lambda_u_val * Lu).item():.4f}, Penalty {penalty.item():.4f}, "
                   f"loss {loss.item():.4f}")

Using device: mps


In [16]:
model1 = DummyModel().to(device)
model2 = DummyModel().to(device)
optimizer = torch.optim.Adam(
    [
        {'params': model1.bert.parameters(), 'lr': 1e-5},
        {'params': model1.classifier.parameters(), 'lr': 1e-4}
    ]
)

labelled_loader = DataLoader(DummyDataset(), batch_size=4, collate_fn=collate_fn)
unlabelled_loader = DataLoader(DummyDataset(), batch_size=4, collate_fn=collate_fn)

num_epochs = 5

for epoch in range(num_epochs):
    train(
        epoch_no=epoch,
        model1=model1,
        model2=model2,
        optimizer=optimizer,
        semiloss=dummy_semiloss,
        labelled_loader=labelled_loader,
        unlabelled_loader=unlabelled_loader,
        warmup_epochs=2,
        batch_size=4,
        device=device
    )


Training:  67%|██████▋   | 2/3 [00:00<00:00, 29.92it/s]


Epoch 0, Batch 1/3, Grad Norms near zero: 0, Lx 0.4727, Lu 0.2677, Penalty 0.0004, loss 0.7408
Epoch 0, Batch 2/3, Grad Norms near zero: 0, Lx 0.1472, Lu 0.7335, Penalty 0.0008, loss 0.8814


Training:  67%|██████▋   | 2/3 [00:00<00:00, 77.65it/s]


Epoch 1, Batch 1/3, Grad Norms near zero: 0, Lx 0.2422, Lu 0.4457, Penalty 0.0001, loss 0.6881
Epoch 1, Batch 2/3, Grad Norms near zero: 0, Lx 0.4451, Lu 0.2013, Penalty 0.0010, loss 0.6474


Training:  67%|██████▋   | 2/3 [00:00<00:00, 66.22it/s]


Epoch 2, Batch 1/3, Grad Norms near zero: 0, Lx 0.2837, Lu 0.1795, Penalty 0.0024, loss 0.4656
Epoch 2, Batch 2/3, Grad Norms near zero: 0, Lx 0.6742, Lu 0.6470, Penalty 0.0071, loss 1.3282


Training:  67%|██████▋   | 2/3 [00:00<00:00, 77.71it/s]


Epoch 3, Batch 1/3, Grad Norms near zero: 0, Lx 0.3017, Lu 0.2343, Penalty 0.0001, loss 0.5361
Epoch 3, Batch 2/3, Grad Norms near zero: 0, Lx 0.3224, Lu 0.1326, Penalty 0.0000, loss 0.4551


Training:  67%|██████▋   | 2/3 [00:00<00:00, 78.82it/s]

Epoch 4, Batch 1/3, Grad Norms near zero: 0, Lx 0.4208, Lu 0.6673, Penalty 0.0001, loss 1.0882
Epoch 4, Batch 2/3, Grad Norms near zero: 0, Lx 0.3638, Lu 0.6945, Penalty 0.0046, loss 1.0629
